# Decorator

In [1]:
from collections.abc import Callable

## Intro

Decorator is structural pattern.

## Practice

### Basic decorator stuff

We have a function which works and maybe just well enough.

In [2]:
def foo(a: float, coef: float = 1, shift: int = 0) -> float:
    return a * coef + shift

foo(1, coef=2, shift=3)

5

Let's say we would like to add custom behaviour that must be in effect ___every time___ the function `foo` is called. The idea is that this original function must not be modified in any way because it is damn critical and any change will result in tedious work.

Example 1. Every time the function is called it should return a string which can be expressed as `"value = {foo(...)}"`. An adapter function could help, yet there is an unavoidable constraint: "every time the function `foo` is called, the new behaviour is in effect". However, an adapter function for `foo` has its own name and replacing `foo` with it is not required even banned.

Here a decorator, or decorating function (callable), comes to the rescue because:
1. it can just wrap the target function;
2. it adds required logic without affecting the target function itself;
3. we can call the target function and it the result will be featured.

In [3]:
FooType = Callable[[float, float, int], float]
# this decorator function is designed to wrap only the `foo` function.
# That is why its type hints are so specific.
def prefix(func: FooType) -> FooType:
    # an inner function that calls a wrappee
    # `func` is local, but it is enclosed to the `wrapper`
    def wrapper(x: float, y: float = 1, z: int = 0) -> str:
        result = func(x, y, z)  # unchaged
        return f"value = {result}"  # add new behaviour
    # when returning wrapper, `func` is enclosed with it,
    # and calling `func` inside the `wrapper` is valid
    return wrapper

# `@decorator` is syntatic sugar over the `foo` function
# it is the same as `foo = prefix(foo)`
@prefix
def foo(a: float, coef: float = 1, shift: int = 0) -> float:
    return a * coef + shift

# foo = prefix(foo)  # care to comment the "@prefix" line first

foo(1, 2, 3)

'value = 5'

Well, it works with major drawbacks. The `prefix` decorator is specific for the `foo` function, so looks fine at first glance...but it does not.

First, this decorator fails to prefix callables which signatures differ.

In [4]:
@prefix
def goo() -> str:
    return "GOO"

# it will not work -> missing one required argument
# and it fails when passed through type checking
goo()

TypeError: prefix.<locals>.wrapper() missing 1 required positional argument: 'x'

Second, Python is powerful enough, so why not defining decorators in a generic way?! Actually we can and let's demonstrate it with another decorator that adds some suffix, so it be `"{foo(...)} -> a good number"`. This time we try `foo = decorator(foo)` syntax in lieu of notorious syntactic sugar. Also, let's rewrite the `prefix` decorator so it can match variadic functions.

In [5]:
def prefix(func: Callable) -> Callable:
    def wrapper(*args, **kwargs):
        # Behold the power of Python!
        result = func(*args, **kwargs)
        return f"value = {result}"
    return wrapper


def suffix(func: Callable) -> Callable:
    def wrapper(*args, **kwargs):
        # Behold the power of Python!
        result = func(*args, **kwargs)
        return f"{result} -> a good number"
    return wrapper


def foo(a: float, coef: float = 1, shift: int = 0) -> float:
    return a * coef + shift

foo = suffix(foo)

foo(1, 2, 3)

'5 -> a good number'

### Order matters

The order in which decorators are applied to a function (callable object) matters. Consider the following examples.

In [6]:
@prefix
@suffix
def twenty_one():
    return 21

@suffix
@prefix
def fourty_two() -> int:
    return 42

print(twenty_one())
print(fourty_two())

value = 21 -> a good number
value = 42 -> a good number


Looks like no difference, so how about this?

In [7]:
def bracketize(func):
    def _(*args, **kwargs):
        return f"[{func(*args, **kwargs)}]"
    return _


def parenthesize(func):
    def _(*args, **kwargs):
        return f"({func(*args, **kwargs)})"
    return _


# actually `*params` is more technically correct
# because they are parametres, not arguments
def goo(*params: float):
    return sum(params)


par_bra = parenthesize(bracketize(goo))
bra_par = bracketize(parenthesize(goo))

print(bra_par(1, 2, 3))
print(par_bra(1, 2, 3))

[(6)]
([6])


Now the order of wrapping is obvious, so this is where subtle mistakes can leak, but you know how to fix them (mere reordering make them disappear gloriously without any magic).